# Whisper ASR

状态：**unverified**。这是单样本推理体验，未经 Colab GPU 实测前不代表验证通过。

模型来源：[官方代码](https://github.com/openai/whisper) · [模型说明](https://github.com/openai/whisper#available-models-and-languages) · 许可：MIT。运行环境：Colab GPU（推荐 T4）。


## 1. 检查 Colab 环境
请在 Runtime → Change runtime type 中选择 GPU。

In [ ]:
import os, platform, shutil, subprocess, sys
from pathlib import Path
assert 'google.colab' in sys.modules or Path('/content').exists(), '请在 Google Colab 中运行'
assert shutil.which('nvidia-smi'), '未检测到 Colab GPU，请切换运行时'
print(subprocess.check_output(['nvidia-smi', '--query-gpu=name,memory.total,driver_version', '--format=csv,noheader'], text=True))
print('Python:', sys.version)
print('Platform:', platform.platform())
print('Disk free GiB:', round(shutil.disk_usage('/content').free / 2**30, 2))


## 2. 安装依赖
重复执行会检查固定版本。保留 Colab 自带 PyTorch，避免替换其 CUDA 组合。

In [ ]:
%pip install -q 'openai-whisper==20250625' 'soundfile>=0.12,<1'


## 3. 记录环境并加载官方模型
权重写入 Colab 临时目录 `/content/models/whisper`，不写入仓库或默认用户缓存。运行时联网下载，官方加载器校验权重 SHA-256。

In [ ]:
import importlib.metadata as metadata
import torch, whisper
MODEL_NAME = 'small'
MODEL_REVISION = 'small（openai-whisper 20250625 发布版本；权重 SHA-256 由官方下载校验）'
MODEL_DIR = Path('/content/models/whisper')
MODEL_DIR.mkdir(parents=True, exist_ok=True)
print('model:', MODEL_NAME, MODEL_REVISION)
for pkg in ('openai-whisper', 'torch', 'soundfile'):
    print(pkg, metadata.version(pkg))
print('CUDA:', torch.version.cuda, 'GPU:', torch.cuda.get_device_name(0))
model = whisper.load_model(MODEL_NAME, device='cuda', download_root=str(MODEL_DIR))
print('model loaded from:', MODEL_DIR)


## 4. 上传音频
上传一条你有权使用的音频。Whisper 官方加载流程通过 ffmpeg 解码并转换为 16 kHz 单声道。

In [ ]:
from google.colab import files
from IPython.display import Audio, display
uploaded = files.upload()
assert len(uploaded) == 1, '请只上传一条音频'
audio_name, audio_bytes = next(iter(uploaded.items()))
audio_path = Path('/content') / Path(audio_name).name
audio_path.write_bytes(audio_bytes)
display(Audio(filename=str(audio_path)))
print('input:', audio_path.name, 'bytes:', audio_path.stat().st_size)


## 5. 转写并记录单次耗时与显存
计时包含该次调用中的音频解码、预处理、GPU 推理和文本后处理，不包含模型加载与上传。单次耗时不代表稳定延迟；无参考转写文本，CER/WER 待测。

In [ ]:
import time
torch.cuda.synchronize()
torch.cuda.reset_peak_memory_stats()
before_reserved = torch.cuda.memory_reserved()
start = time.perf_counter()
result = model.transcribe(str(audio_path), fp16=True)
torch.cuda.synchronize()
elapsed_s = time.perf_counter() - start
print('转写:', result['text'])
print('识别语言:', result['language'])
print('单次耗时 s:', round(elapsed_s, 3))
print('GPU 峰值 allocated MiB:', round(torch.cuda.max_memory_allocated() / 2**20, 1))
print('GPU 峰值 reserved MiB:', round(torch.cuda.max_memory_reserved() / 2**20, 1))
print('调用前 reserved MiB:', round(before_reserved / 2**20, 1))
print('CER/WER: 待测（无参考文本）')


## 结果边界
当前仓库验证状态：**unverified**。保存真实 Colab GPU 完整运行日志后才能更新状态。Colab 临时文件会随会话消失；请自行下载需要保留的结果。